# Recent Crossmatches Demo

Query the SCiMMA Rubin Crossmatch Service for the catalog crossmatches of
every object with an alert in a recent time window, grouped by object.

The endpoint is a read-only HTTP GET and is public and unauthenticated, so no
credentials are needed today. **Authentication and authorization will be
required in a future release** -- no date or mechanism is fixed yet, so write
client code that can take credentials later without restructuring it. See
`docs/api/recent-crossmatch-api.md` for the full parameter reference.

Results are **keyset (cursor) paged**: each response returns one page of objects
plus an opaque `next_cursor`. Following the cursor to exhaustion pulls the whole
window (section 4). This notebook runs top to bottom against the production
endpoint. Point `BASE_URL` at a local `gunicorn`/`runserver` instance to run it
against a checkout instead.

In [ ]:
import math
import random
import time

import httpx
import pandas as pd

# Production host. For a local server use e.g. 'http://127.0.0.1:8000'.
BASE_URL = 'https://crossmatch.scimma.org'
ENDPOINT = f'{BASE_URL}/api/recent-crossmatches'

# Bounded retry for the edge rate limit. Only 429 is retried; every other status
# surfaces immediately, so a malformed cursor still fails fast.
RATE_LIMIT_STATUS = 429
MAX_ATTEMPTS = 5
INITIAL_DELAY = 1.0
MAX_DELAY = 30.0


def _retry_delay(attempt, retry_after):
    """Seconds to wait before retry `attempt` (1-based), bounded on both ends.

    A numeric Retry-After is honored, but floored at INITIAL_DELAY and capped at
    MAX_DELAY: a negative value (a proxy whose clock is behind) or a zero would
    otherwise crash time.sleep or collapse the backoff into a burst aimed at an
    edge that just reported overload. Missing, malformed, non-finite, and
    HTTP-date values fall back to exponential backoff. Both paths add jitter so
    simultaneous readers do not retry in lockstep -- the honored path only ever
    waits longer than the server asked, never less.
    """
    honored = None
    if retry_after is not None:
        try:
            secs = float(retry_after)
        except (TypeError, ValueError):
            secs = None  # HTTP-date or junk -- use the computed backoff.
        if secs is not None and math.isfinite(secs):
            honored = max(INITIAL_DELAY, min(secs, MAX_DELAY))
    if honored is not None:
        return min(honored + random.random(), MAX_DELAY)
    delay = min(INITIAL_DELAY * (2 ** (attempt - 1)), MAX_DELAY)
    return delay * (0.5 + random.random() / 2)  # jitter: 50-100% of delay


def get_recent(**params):
    """GET the endpoint with the given query params and return the parsed JSON.

    Retries only on HTTP 429 -- the per-source-IP edge rate limit -- with
    exponential backoff and jitter, up to MAX_ATTEMPTS. Every other error raises
    on the first attempt, and a 429 that survives the last attempt raises too.
    """
    for attempt in range(1, MAX_ATTEMPTS + 1):
        resp = httpx.get(ENDPOINT, params=params, timeout=60.0)
        if resp.status_code == RATE_LIMIT_STATUS and attempt < MAX_ATTEMPTS:
            delay = _retry_delay(attempt, resp.headers.get('Retry-After'))
            print(f'rate limited; retrying in {delay:.1f}s '
                  f'(attempt {attempt}/{MAX_ATTEMPTS})')
            time.sleep(delay)
            continue
        resp.raise_for_status()
        return resp.json()

## 1. Default window and `matches` detail

No parameters: the last 12 hours (by `ingest_time`), with each object's
position and its catalog matches (catalog, source id, separation). `count` is
the number of objects **on this page**; `next_cursor` is non-null when more
pages remain (section 4 walks them all).

In [ ]:
result = get_recent()
print('window:', result['window'])
print('time_field:', result['time_field'], '| detail:', result['detail'])
print('page_size:', result['page_size'])
print('objects on this page:', result['count'])
print('more pages?', result['next_cursor'] is not None)

### Flatten the matches into a DataFrame

One row per (object, match): the object id and position alongside each match's
catalog, source id, and separation.

In [ ]:
rows = []
for obj in result['objects']:
    for match in obj.get('matches', []):
        rows.append({
            'diaObjectId': obj['diaObjectId'],
            'object_ra': obj.get('ra'),
            'object_dec': obj.get('dec'),
            'catalog_name': match['catalog_name'],
            'catalog_source_id': match['catalog_source_id'],
            'separation_arcsec': match['separation_arcsec'],
        })

matches_df = pd.DataFrame(rows)
matches_df

In [ ]:
# How many matches per catalog across all objects in the window?
if not matches_df.empty:
    display(matches_df['catalog_name'].value_counts())

## 2. An explicit window

Pass ISO-8601 UTC `start`/`end`. Here, the last 24 hours by observation time
(`event_time`) instead of ingest time.

In [ ]:
from datetime import datetime, timedelta, timezone

end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)

windowed = get_recent(
    start=start.isoformat(),
    end=end.isoformat(),
    time_field='event_time',
)
print('objects in the last 24h (by event_time):', windowed['count'])

## 3. Full published payload

`detail=full` returns, for each match, exactly the payload the service
publishes over Hopskotch — including the nested `catalog_payload` of
catalog-specific columns. The `ra`/`dec` inside a match are the matched catalog
**source** coordinates, distinct from the object position.

In [ ]:
full = get_recent(detail='full', page_size=100)
print('objects on this page:', full['count'])

# Show the first full match payload, if any.
for obj in full['objects']:
    if obj.get('matches'):
        first_match = obj['matches'][0]
        break
else:
    first_match = None

first_match

In [ ]:
# Normalize the full payloads into a wide DataFrame: top-level match fields plus
# the flattened catalog_payload columns.
full_rows = []
for obj in full['objects']:
    for match in obj.get('matches', []):
        row = {k: v for k, v in match.items() if k != 'catalog_payload'}
        row.update(match.get('catalog_payload') or {})
        full_rows.append(row)

full_df = pd.DataFrame(full_rows)
full_df.head()

## 4. Page an entire window

A single request returns at most one `page_size` page. To pull an *entire*
window (for example, last night's full set of matched objects), follow
`next_cursor` until it comes back `null`. The cursor pins the query context, so
after the first request you pass **only** the cursor (and optionally a
`page_size`); repeating `start`/`end`/`time_field`/`detail` with a different
value would be rejected.

In [ ]:
def iter_all_objects(page_size=1000, **params):
    """Yield every object across all pages for a query, following next_cursor.

    Pass the same query params you'd pass to a single request (window,
    time_field, detail). After the first page only the cursor is sent, since the
    cursor pins the query context.

    Rate-limit (429) handling lives in `get_recent`, so this stays a thin cursor
    walk -- each page inherits the same bounded retry.
    """
    cursor = None
    while True:
        page = get_recent(cursor=cursor, page_size=page_size) if cursor \
            else get_recent(page_size=page_size, **params)
        yield from page['objects']
        cursor = page['next_cursor']
        if cursor is None:
            break


# Walk last night's window (by event_time) at the `matches` level, small pages so
# the loop actually iterates even on modest data.
from datetime import datetime, timedelta, timezone

end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)

all_objects = list(iter_all_objects(
    page_size=500,
    start=start.isoformat(),
    end=end.isoformat(),
    time_field='event_time',
    detail='matches',
))
print('total distinct objects pulled across all pages:', len(all_objects))

In [ ]:
# Flatten the full paged pull into one DataFrame (one row per object/match), and
# confirm every diaObjectId is distinct and appears exactly once.
paged_rows = []
for obj in all_objects:
    for match in obj.get('matches', []):
        paged_rows.append({
            'diaObjectId': obj['diaObjectId'],
            'catalog_name': match['catalog_name'],
            'catalog_source_id': match['catalog_source_id'],
            'separation_arcsec': match['separation_arcsec'],
        })

paged_df = pd.DataFrame(paged_rows)
n_distinct = paged_df['diaObjectId'].nunique() if not paged_df.empty else 0
print('distinct diaObjectIds across the whole window:', n_distinct)
print('one page would have returned at most page_size objects; paging returns them all')
paged_df.head()